# 3 · Probabilités et mesure de la qualité — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch03_probabilites/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch03_probabilites/03_notebook.ipynb`, créée par `python tools/start_chapter.py 3`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 3.1, 3.2, 3.3, 3.5, 3.6, 3.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 3.12 à 3.14 | probabilités par les aires et les tables : fléchettes simulées, deux disques, pd.crosstab (*prochaine session de génération*) | | | |
| B | 3.15 à 3.19 | la matrice de confusion et les mesures binaires dans mylearn.metrics (*prochaine session de génération*) | | | |
| C | 3.20 à 3.23 | seuil de décision, prévalence, outils et documentation de scikit-learn (*prochaine session de génération*) | | | |
| D | 3.24 à 3.29 | au-delà du livre : courbes ROC et precision-recall, moyennes sur plusieurs classes, calibration, défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="3")  # your own library

## Objectifs et rappel express

- Calculer des probabilités simples, conditionnelles, jointes et marginales, et ne pas confondre $P(A \mid B)$ avec $P(B \mid A)$.
- Construire une matrice de confusion et en tirer accuracy, precision, recall, F1 et les autres mesures, avec les conventions de scikit-learn.
- Choisir une mesure et un seuil selon le coût des erreurs et la prévalence.
- Évaluer des scores sur tous les seuils (courbes ROC et precision-recall) et vérifier des probabilités (calibration).

**Rappel express.** $P(A \mid B) = \frac{P(A, B)}{P(B)}$ ; $P(A, B) = P(A \mid B)\,P(B)$ ; matrice de confusion de scikit-learn pour des étiquettes 0/1 : `[[TN, FP], [FN, TP]]` ; accuracy $= \frac{TP + TN}{n}$ ; precision $= \frac{TP}{TP + FP}$ ; recall $= \frac{TP}{TP + FN}$ ; spécificité $= \frac{TN}{TN + FP}$ ; $F_1 = \frac{2\,TP}{2\,TP + FP + FN}$. En Python : `pd.crosstab`, `sklearn.metrics.confusion_matrix`, `classification_report`, `roc_curve`, `precision_recall_curve`.

## Partie 0 · Vérifier tes exercices papier (✏️ 3.1, 3.2, 3.3, 3.5, 3.6 et 3.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 3.4 et 3.8 se corrigent avec `05_solutions.md`.

**Ex 3.1 — Fléchettes et aires : probabilités simples et conditionnelles**

In [ ]:
# Ex 3.1: the values you found on paper
answer_3_1a = ...  # a) P(A)
answer_3_1b = ...  # b) P(B)
answer_3_1c = ...  # c) P(A, B)
answer_3_1d = ...  # d) P(A | B)
answer_3_1e = ...  # e) P(B | A), 3 decimals
answer_3_1f = ...  # f) P(neither A nor B)
answer_3_1g = ...  # g) expected darts in B out of 400
answer_3_1h = ...  # h) expected darts in A and B out of 400
answer_3_1i = ...  # i) the student's estimate of P(A | B), 3 decimals
answer_3_1j = ...  # j) are A and B independent? True or False

for letter, answer in zip("abcdefghij", [answer_3_1a, answer_3_1b, answer_3_1c, answer_3_1d, answer_3_1e, answer_3_1f, answer_3_1g, answer_3_1h, answer_3_1i, answer_3_1j]):
    wb.check(f"3.1{letter}", answer)

**Ex 3.2 — Les 20 points : matrice de confusion et quatre mesures**

In [ ]:
# Ex 3.2: the values you found on paper
answer_3_2a = ...  # a) TP
answer_3_2b = ...  # b) FP
answer_3_2c = ...  # c) FN
answer_3_2d = ...  # d) TN
answer_3_2e = ...  # e) the confusion matrix as scikit-learn returns it, a list of 2 rows
answer_3_2f = ...  # f) the accuracy, 2 decimals
answer_3_2g = ...  # g) the precision, 3 decimals
answer_3_2h = ...  # h) the recall, 2 decimals
answer_3_2i = ...  # i) the F1 score, 3 decimals

for letter, answer in zip("abcdefghi", [answer_3_2a, answer_3_2b, answer_3_2c, answer_3_2d, answer_3_2e, answer_3_2f, answer_3_2g, answer_3_2h, answer_3_2i]):
    wb.check(f"3.2{letter}", answer)

**Ex 3.3 — Le glacier : jointes, marginales et conditionnelles**

In [ ]:
# Ex 3.3: the values you found on paper
answer_3_3a = ...  # a) P(V)
answer_3_3b = ...  # b) P(C), 3 decimals
answer_3_3c = ...  # c) P(V, C)
answer_3_3d = ...  # d) P(V | C), 3 decimals
answer_3_3e = ...  # e) P(C | V)
answer_3_3f = ...  # f) P(chocolate | cup), 3 decimals
answer_3_3g = ...  # g) are the flavour and the container independent? True or False
answer_3_3h = ...  # h) vanilla cones to plan for 300 customers

for letter, answer in zip("abcdefgh", [answer_3_3a, answer_3_3b, answer_3_3c, answer_3_3d, answer_3_3e, answer_3_3f, answer_3_3g, answer_3_3h]):
    wb.check(f"3.3{letter}", answer)

**Ex 3.5 — Toutes les mesures du tableau récapitulatif**

In [ ]:
# Ex 3.5: the values you found on paper
answer_3_5a = ...  # a) the prevalence, 3 decimals
answer_3_5b = ...  # b) the accuracy, 3 decimals
answer_3_5c = ...  # c) the precision, 3 decimals
answer_3_5d = ...  # d) the recall, 3 decimals
answer_3_5e = ...  # e) the specificity, 3 decimals
answer_3_5f = ...  # f) the NPV, 3 decimals
answer_3_5g = ...  # g) the FPR, 3 decimals
answer_3_5h = ...  # h) the FNR, 3 decimals
answer_3_5i = ...  # i) the FDR, 3 decimals
answer_3_5j = ...  # j) the FOR (false omission rate), 3 decimals
answer_3_5k = ...  # k) the F1 score, 3 decimals
answer_3_5l = ...  # l) the balanced accuracy, 3 decimals
answer_3_5m = ...  # m) the MCC, 3 decimals

for letter, answer in zip("abcdefghijklm", [answer_3_5a, answer_3_5b, answer_3_5c, answer_3_5d, answer_3_5e, answer_3_5f, answer_3_5g, answer_3_5h, answer_3_5i, answer_3_5j, answer_3_5k, answer_3_5l, answer_3_5m]):
    wb.check(f"3.5{letter}", answer)

**Ex 3.6 — Trois espèces : moyennes macro, micro et pondérée**

In [ ]:
# Ex 3.6: the values you found on paper
answer_3_6a = ...  # a) the support of each species, a list
answer_3_6b = ...  # b) the precision of each species, a list (3 decimals)
answer_3_6c = ...  # c) the recall of each species, a list (3 decimals)
answer_3_6d = ...  # d) the F1 score of each species, a list (3 decimals)
answer_3_6e = ...  # e) the macro precision, 3 decimals
answer_3_6f = ...  # f) the macro F1 score, 3 decimals
answer_3_6g = ...  # g) the weighted F1 score, 3 decimals
answer_3_6h = ...  # h) the micro F1 score, 3 decimals

for letter, answer in zip("abcdefgh", [answer_3_6a, answer_3_6b, answer_3_6c, answer_3_6d, answer_3_6e, answer_3_6f, answer_3_6g, answer_3_6h]):
    wb.check(f"3.6{letter}", answer)

**Ex 3.7 — Le test « fiable à 99 % » dans une ville à 1 % de malades**

In [ ]:
# Ex 3.7: the values you found on paper
answer_3_7a = ...  # a) the number of sick people
answer_3_7b = ...  # b) TP
answer_3_7c = ...  # c) FN
answer_3_7d = ...  # d) FP
answer_3_7e = ...  # e) TN
answer_3_7f = ...  # f) the precision P(sick | positive), 2 decimals
answer_3_7g = ...  # g) the NPV P(healthy | negative), 4 decimals
answer_3_7h = ...  # h) the accuracy, 2 decimals
answer_3_7i = ...  # i) the precision with a prevalence of 0.2 %, 3 decimals
answer_3_7j = ...  # j) the precision with a prevalence of 10 %, 3 decimals
answer_3_7k = ...  # k) the book's specificity TN / (TN + FP), 2 decimals
answer_3_7l = ...  # l) the book's NPV TN / (TN + FN), 4 decimals

for letter, answer in zip("abcdefghijkl", [answer_3_7a, answer_3_7b, answer_3_7c, answer_3_7d, answer_3_7e, answer_3_7f, answer_3_7g, answer_3_7h, answer_3_7i, answer_3_7j, answer_3_7k, answer_3_7l]):
    wb.check(f"3.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer une probabilité conditionnelle dans une table, et dire laquelle de $P(A \mid B)$ ou $P(B \mid A)$ répond à une question ?
2. Sais-tu construire une matrice de confusion et choisir entre precision et recall selon le coût des erreurs ?
3. Sais-tu expliquer pourquoi un bon test donne surtout des faux positifs quand la maladie est rare ?

**Pour aller plus loin** : les pages du *Machine Learning Crash Course* de Google et le guide « Metrics and scoring » de scikit-learn, cités dans la fiche. La suite : le ch. 4 (règle de Bayes), qui passe de $P(\text{positif} \mid \text{malade})$ à $P(\text{malade} \mid \text{positif})$ ; `mylearn.metrics` servira dans tous les chapitres suivants.

*Les exercices de code 3.12 à 3.29 (parties A à D, ta librairie `mylearn.metrics`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 3` pour obtenir le notebook complet.*